# sipl-formal: демонстрація

In [1]:
from sipl_formal.grammar import parse_grammar
from sipl_formal.normal_form import to_cnf, steps_to_text
from sipl_formal.equations import report
from sipl_formal.pumping import parse_pattern, explore, exploration_to_text, pump, split_by_lengths, split_to_text
from sipl_formal.derivation import find_derivations, result_to_text
from sipl_formal.sipl import parse, tree_to_text, SiplSyntaxError

## FORM-01: нормальна форма Хомського

In [2]:
g = parse_grammar("""
S -> AbBC | ADcBC | ADCC | ACc
C -> eps | bCc
A -> eps | aA
B -> eps | bB
D -> dD
""")
cnf, steps = to_cnf(g)
print(steps_to_text(g, steps, cnf))

Вхідна граматика:
S → AbBC | ADcBC | ADCC | ACc
C → ε | bCc
A → ε | aA
B → ε | bB
D → dD

Крок 1. Видалення непродуктивних нетерміналів
  Pr0 = {C, A, B}
  Pr1 = {C, A, B, S}
  Множина стабілізувалась: Pr = {C, A, B, S}.
  Непродуктивні: {D}.
  видалено: S → ADcBC, S → ADCC, D → dD

Крок 2. Видалення недосяжних нетерміналів
  R0 = {S}
  R1 = {S, A, B, C}
  Недосяжних нетерміналів немає.

Крок 3. Видалення ε-правил
  ε-нетермінали (з них виводиться ε): {C, A, B}.
  з S → AbBC додано: AbB, AbC, Ab, bBC, bB, bC, b
  з S → ACc додано: Ac, Cc, c
  з C → bCc додано: bc
  з A → aA додано: a
  з B → bB додано: b
  видалено: C → ε, A → ε, B → ε
  Аксіома S не є ε-нетерміналом, тому ε не належить мові.

Крок 4. Видалення ланцюгових правил (A → B)
  Ланцюгових правил немає.

Крок 5. Заміна терміналів у довгих правилах новими нетерміналами
  B′ → b
  C′ → c
  A′ → a

Крок 6. Розбиття правил, де справа більше двох нетерміналів
  S → AB′BC: S → AD, D → B′E, E → BC
  S → AB′B: S → AF, F → B′B
  S → A

### Новий приклад

In [3]:
g = parse_grammar("""
S -> aSbS | eps
""")
cnf, steps = to_cnf(g)
print(steps_to_text(g, steps, cnf))

Вхідна граматика:
S → aSbS | ε

Крок 1. Видалення непродуктивних нетерміналів
  Pr0 = {S}
  Множина стабілізувалась: Pr = {S}.
  Непродуктивних нетерміналів немає.

Крок 2. Видалення недосяжних нетерміналів
  R0 = {S}
  Недосяжних нетерміналів немає.

Крок 3. Видалення ε-правил
  ε-нетермінали (з них виводиться ε): {S}.
  з S → aSbS додано: aSb, abS, ab
  видалено: S → ε
  ε належить мові, тому введено нову аксіому S′: S′ → S | ε.

Крок 4. Видалення ланцюгових правил (A → B)
  S′ ⇒* S, тому S′ отримує правила S: aSbS, aSb, abS, ab
  видалено: S′ → S

Крок 5. Заміна терміналів у довгих правилах новими нетерміналами
  A′ → a
  B′ → b

Крок 6. Розбиття правил, де справа більше двох нетерміналів
  S′ → A′SB′S: S′ → A′D, D → SE, E → B′S
  S′ → A′SB′: S′ → A′F, F → SB′
  S′ → A′B′S: S′ → A′E (E → … вже є)
  S → A′SB′S: S → A′D (D → … вже є)
  S → A′SB′: S → A′F (F → … вже є)
  S → A′B′S: S → A′E (E → … вже є)

Нормальна форма Хомського:
S′ → ε | A′D | A′F | A′E | A′B′
D → SE
E → B′S
F → SB′


## FORM-02: система рівнянь і наближення

In [4]:
g = parse_grammar("""
S -> aAcc
A -> eeAd | b
""")
print(report(g, max_len=14))

Система рівнянь:
S = {a}A{cc}
A = {ee}A{d} ∪ {b}

Наближення (слова довжини ≤ 14):
S0 = ∅, A0 = ∅
S1 = ∅, A1 = {b}
S2 = {abcc}, A2 = {b, eebd}
S3 = {abcc, aeebdcc}, A3 = {b, eebd, eeeebdd}
S4 = {abcc, aeebdcc, aeeeebddcc}, A4 = {b, eebd, eeeebdd, eeeeeebddd}
S5 = {abcc, aeebdcc, aeeeebddcc, aeeeeeebdddcc}, A5 = {b, eebd, eeeebdd, eeeeeebddd, eeeeeeeebdddd}
Стабілізація на кроці 5 (на словах довжини ≤ 14).

Розв'язок для S: {abcc, aeebdcc, aeeeebddcc, aeeeeeebdddcc}
Збігається з мовою граматики на словах довжини ≤ 14.


### Новий приклад

In [5]:
g = parse_grammar("""
S -> (S)S | eps
""")
print(report(g, max_len=6))

Система рівнянь:
S = {(}S{)}S ∪ {ε}

Наближення (слова довжини ≤ 6):
S0 = ∅
S1 = {ε}
S2 = {ε, ()}
S3 = {ε, (), (()), ()(), (())()}
S4 = {ε, (), (()), ()(), ((())), (()()), (())(), ()(()), ()()()}
Стабілізація на кроці 4 (на словах довжини ≤ 6).

Розв'язок для S: {ε, (), (()), ()(), ((())), (()()), (())(), ()(()), ()()()}
Збігається з мовою граматики на словах довжини ≤ 6.


## FORM-03: Pumping Lemma Explorer:

In [6]:
lang = parse_pattern("a e^{2n} b^n d^n c^{2n}")
print(exploration_to_text(explore(lang, k=4, n=4), show=2))

Мова: L = { a e^{2n} b^n d^n c^{2n} | n ≥ 0 }
Слово: t = aeeeeeeeebbbbddddcccccccc (n = 4, |t| = 25), k = 4
Перевірено розкладів: 685; i = 0…3
Жоден із 685 розкладів не витримує накачки: для кожного є i ≤ 3, при якому слово виходить за межі мови. Для цього слова й k лема не виконується.

Розклади, що не витримують, по одному на кожен випадок:

Випадок: t1 або t2 зачіпає блок сталої довжини, тож при повторенні цей блок змінюється.
t = u[t1]x[t2]v = [a][e]eeeeeeebbbbddddcccccccc
u = «», t1 = «a», x = «», t2 = «e», v = «eeeeeeebbbbddddcccccccc»
|t1t2| = 2 ≥ 1: так
|t1xt2| = 2 ≤ k = 4: так
  i = 0: eeeeeeebbbbddddcccccccc  ∉ L
  i = 1: aeeeeeeeebbbbddddcccccccc  ∈ L
  i = 2: aaeeeeeeeeebbbbddddcccccccc  ∉ L
  i = 3: aaaeeeeeeeeeebbbbddddcccccccc  ∉ L

Випадок: у t1 або t2 різні букви, тож при повторенні ламається порядок букв.
t = u[t1]x[t2]v = aeeeee[e][eeb]bbbddddcccccccc
u = «aeeeee», t1 = «e», x = «», t2 = «eeb», v = «bbbddddcccccccc»
|t1t2| = 4 ≥ 1: так
|t1xt2| = 4 ≤ k = 4: так
  i = 

### Один розклад

In [7]:
lang = parse_pattern("a e^{2n} b^n d^n c^{2n}")
word = lang.word(2)
split = pump(lang, split_by_lengths(word, [1, 2, 0, 1]), i_max=2)  # |u|, |t1|, |x|, |t2|
print(split_to_text(split, k=4))

t = u[t1]x[t2]v = a[ee][e]ebbddcccc
u = «a», t1 = «ee», x = «», t2 = «e», v = «ebbddcccc»
|t1t2| = 3 ≥ 1: так
|t1xt2| = 3 ≤ k = 4: так
  i = 0: aebbddcccc  ∉ L
  i = 1: aeeeebbddcccc  ∈ L
  i = 2: aeeeeeeebbddcccc  ∉ L


### Новий приклад

In [8]:
lang = parse_pattern("a^n b^n")
print(exploration_to_text(explore(lang, k=4, n=4), show=2))

Мова: L = { a^n b^n | n ≥ 0 }
Слово: t = aaaabbbb (n = 4, |t| = 8), k = 4
Перевірено розкладів: 175; i = 0…3
Знайдено 7 розклад(ів), що витримують накачку для i = 0…3. Для цього слова й k лему не спростовано.

Розклади, що витримують накачку:
t = u[t1]x[t2]v = a[a]aa[b]bbb
u = «a», t1 = «a», x = «aa», t2 = «b», v = «bbb»
|t1t2| = 2 ≥ 1: так
|t1xt2| = 4 ≤ k = 4: так
  i = 0: aaabbb  ∈ L
  i = 1: aaaabbbb  ∈ L
  i = 2: aaaaabbbbb  ∈ L
  i = 3: aaaaaabbbbbb  ∈ L
t = u[t1]x[t2]v = aa[a]a[b]bbb
u = «aa», t1 = «a», x = «a», t2 = «b», v = «bbb»
|t1t2| = 2 ≥ 1: так
|t1xt2| = 3 ≤ k = 4: так
  i = 0: aaabbb  ∈ L
  i = 1: aaaabbbb  ∈ L
  i = 2: aaaaabbbbb  ∈ L
  i = 3: aaaaaabbbbbb  ∈ L

Розклади, що не витримують, по одному на кожен випадок:

Випадок: у t1 або t2 різні букви, тож при повторенні ламається порядок букв.
t = u[t1]x[t2]v = a[a][aab]bbb
u = «a», t1 = «a», x = «», t2 = «aab», v = «bbb»
|t1t2| = 4 ≥ 1: так
|t1xt2| = 4 ≤ k = 4: так
  i = 0: abbb  ∉ L
  i = 1: aaaabbbb  ∈ L
  i = 2: aaaa

## FORM-04: виводи і дерева

In [9]:
g = parse_grammar("E -> E+E | E*E | (E) | x")
print(result_to_text(find_derivations(g, "x+x*x")))

Слово: x+x*x
Знайдено різних дерев виводу: 2. Граматика неоднозначна.

— Вивід 1 —
Лівосторонній: E ⇒ E+E ⇒ x+E ⇒ x+E*E ⇒ x+x*E ⇒ x+x*x
Правосторонній: E ⇒ E+E ⇒ E+E*E ⇒ E+E*x ⇒ E+x*x ⇒ x+x*x
Дерево:
E
├─ E
│  └─ x
├─ +
└─ E
   ├─ E
   │  └─ x
   ├─ *
   └─ E
      └─ x

— Вивід 2 —
Лівосторонній: E ⇒ E*E ⇒ E+E*E ⇒ x+E*E ⇒ x+x*E ⇒ x+x*x
Правосторонній: E ⇒ E*E ⇒ E*x ⇒ E+E*x ⇒ E+x*x ⇒ x+x*x
Дерево:
E
├─ E
│  ├─ E
│  │  └─ x
│  ├─ +
│  └─ E
│     └─ x
├─ *
└─ E
   └─ x


### Новий приклад

In [10]:
g = parse_grammar("S -> aSb | ab")
print(result_to_text(find_derivations(g, "aaabbb")))

Слово: aaabbb
Слово має одне дерево виводу (у межах глибини 30).

— Вивід 1 —
Лівосторонній: S ⇒ aSb ⇒ aaSbb ⇒ aaabbb
Правосторонній: S ⇒ aSb ⇒ aaSbb ⇒ aaabbb
Дерево:
S
├─ a
├─ S
│  ├─ a
│  ├─ S
│  │  ├─ a
│  │  └─ b
│  └─ b
└─ b


## CORE-01 і VIS-01: SIPL

In [11]:
program = """begin
  R := 1;
  while Y > 0 do
  begin
    R := R * X;
    Y := Y - 1
  end
end
"""
print(tree_to_text(parse(program)))

<програма>  [1:1–8:4]
├─ begin  [1:1–1:6]
├─ <оператор>  [2:3–7:6] «R := 1; while Y > 0 do begin R := R * X; Y := Y - 1 end»
│  ├─ <оператор>  [2:3–2:9] «R := 1»
│  │  ├─ <змінна>  [2:3–2:4] «R»
│  │  │  └─ R  [2:3–2:4]
│  │  ├─ :=  [2:5–2:7]
│  │  └─ <число>  [2:8–2:9] «1»
│  │     └─ 1  [2:8–2:9]
│  ├─ ;  [2:9–2:10]
│  └─ <оператор>  [3:3–7:6] «while Y > 0 do begin R := R * X; Y := Y - 1 end»
│     ├─ while  [3:3–3:8]
│     ├─ <умова>  [3:9–3:14] «Y > 0»
│     │  ├─ <змінна>  [3:9–3:10] «Y»
│     │  │  └─ Y  [3:9–3:10]
│     │  ├─ >  [3:11–3:12]
│     │  └─ <число>  [3:13–3:14] «0»
│     │     └─ 0  [3:13–3:14]
│     ├─ do  [3:15–3:17]
│     └─ <оператор>  [4:3–7:6] «begin R := R * X; Y := Y - 1 end»
│        ├─ begin  [4:3–4:8]
│        ├─ <оператор>  [5:5–6:15] «R := R * X; Y := Y - 1»
│        │  ├─ <оператор>  [5:5–5:15] «R := R * X»
│        │  │  ├─ <змінна>  [5:5–5:6] «R»
│        │  │  │  └─ R  [5:5–5:6]
│        │  │  ├─ :=  [5:7–5:9]
│        │  │  └─ <вираз>  [5:10–5:15] «

### Програма з помилкою

In [12]:
program = """begin
  R := 1
  Y := 2
end
"""
try:
    parse(program)
except SiplSyntaxError as error:
    print(error)

рядок 3, стовпчик 3: неочікуване «Y», очікувалось: *, +, -, ;, end


### Новий приклад

In [13]:
program = """begin
  while not (X = Y) do
    if X > Y then X := X - Y else Y := Y - X
end
"""
try:
    print(tree_to_text(parse(program)))
except SiplSyntaxError as error:
    print(error)

<програма>  [1:1–4:4]
├─ begin  [1:1–1:6]
├─ <оператор>  [2:3–3:45] «while not (X = Y) do if X > Y then X := X - Y else Y := Y - X»
│  ├─ while  [2:3–2:8]
│  ├─ <умова>  [2:9–2:20] «not (X = Y)»
│  │  ├─ not  [2:9–2:12]
│  │  └─ <умова>  [2:13–2:20] «(X = Y)»
│  │     ├─ (  [2:13–2:14]
│  │     ├─ <умова>  [2:14–2:19] «X = Y»
│  │     │  ├─ <змінна>  [2:14–2:15] «X»
│  │     │  │  └─ X  [2:14–2:15]
│  │     │  ├─ =  [2:16–2:17]
│  │     │  └─ <змінна>  [2:18–2:19] «Y»
│  │     │     └─ Y  [2:18–2:19]
│  │     └─ )  [2:19–2:20]
│  ├─ do  [2:21–2:23]
│  └─ <оператор>  [3:5–3:45] «if X > Y then X := X - Y else Y := Y - X»
│     ├─ if  [3:5–3:7]
│     ├─ <умова>  [3:8–3:13] «X > Y»
│     │  ├─ <змінна>  [3:8–3:9] «X»
│     │  │  └─ X  [3:8–3:9]
│     │  ├─ >  [3:10–3:11]
│     │  └─ <змінна>  [3:12–3:13] «Y»
│     │     └─ Y  [3:12–3:13]
│     ├─ then  [3:14–3:18]
│     ├─ <оператор>  [3:19–3:29] «X := X - Y»
│     │  ├─ <змінна>  [3:19–3:20] «X»
│     │  │  └─ X  [3:19–3:20]
│     │  ├─ :